In [1]:
import pandas as pd
import numpy as np

from google.colab import files

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Saving arcana_financial_data_quality_v2.csv to arcana_financial_data_quality_v2 (1).csv
Dataset loaded successfully!
Rows: 7,714
Columns: 12


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7714 entries, 0 to 7713
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   record_id            7714 non-null   int64  
 1   date                 7714 non-null   object 
 2   company              7714 non-null   object 
 3   ticker               7714 non-null   object 
 4   sector               7714 non-null   object 
 5   provider             7714 non-null   object 
 6   currency             7714 non-null   object 
 7   share_price          7655 non-null   float64
 8   market_cap           7662 non-null   float64
 9   revenue              7653 non-null   float64
 10  portfolio_value      7658 non-null   float64
 11  ingestion_timestamp  7714 non-null   object 
dtypes: float64(4), int64(1), object(7)
memory usage: 723.3+ KB


In [3]:
df.head()

,record_id,date,company,ticker,sector,provider,currency,share_price,market_cap,revenue,portfolio_value,ingestion_timestamp
0,89,2026-08-01,AMD,AMD,Technology,Provider_A,USD,102.58,1.468768e+12,4.041251e+11,79983893.0,2026-09-08 22:03:45
1,7679,2026-08-01,AMD,AMD,Technology,Provider_A,USD,102.58,1.468768e+12,4.041251e+11,79983893.0,2026-09-08 22:03:45
2,90,2026-08-01,AMD,AMD,Technology,Provider_B,USD,102.35,1.465488e+12,4.032224e+11,79805246.0,2026-09-08 23:49:47
3,91,2026-08-01,AMD,AMD,Technology,Provider_C,USD,102.23,1.463795e+12,4.027568e+11,79713083.0,2026-09-08 13:41:37
4,92,2026-08-01,AMD,AMD,Technology,Provider_D,USD,102.38,1.465862e+12,4.033254e+11,79825624.0,2026-09-08 09:57:50


In [4]:
df.isna().sum()

,0
record_id,0
date,0
company,0
ticker,0
sector,0
provider,0
currency,0
share_price,59
market_cap,52
revenue,61


In [5]:
# Check 1: Missing values

missing = df.isna().sum().sort_values(ascending=False)

missing_pct = (missing / len(df) * 100).round(2)

completeness_report = pd.DataFrame({
    "Missing_Count": missing,
    "Missing_Percentage": missing_pct
})

completeness_report

,Missing_Count,Missing_Percentage
revenue,61,0.79
share_price,59,0.76
portfolio_value,56,0.73
market_cap,52,0.67
ticker,0,0.00
company,0,0.00
date,0,0.00
record_id,0,0.00
currency,0,0.00
provider,0,0.00


In [6]:
# Records containing at least one missing value

missing_records = df[df.isna().any(axis=1)]

print(f"Records with missing values: {len(missing_records):,}")
print(f"Percentage of records affected: {len(missing_records) / len(df) * 100:.2f}%")

Records with missing values: 228
Percentage of records affected: 2.96%


In [7]:
missing_records.head(10)

,record_id,date,company,ticker,sector,provider,currency,share_price,market_cap,revenue,portfolio_value,ingestion_timestamp
53,56,2026-08-01,Chevron Corp.,CVX,Energy,Provider_D,USD,193.72,NaN,3.941171e+11,171902370.0,2026-09-08 11:38:19
70,145,2026-08-01,Duke Energy,DUK,Utilities,Provider_A,USD,NaN,2.995150e+11,3.912094e+11,73799904.0,2026-09-08 18:55:23
77,52,2026-08-01,Exxon Mobil,XOM,Energy,Provider_D,USD,46.55,1.402302e+12,9.628089e+10,NaN,2026-09-08 01:28:25
94,45,2026-08-01,Johnson & Johnson,JNJ,Healthcare,Provider_A,USD,604.79,NaN,2.920738e+11,61000288.0,2026-09-08 19:11:27
114,113,2026-08-01,Merck & Co.,MRK,Healthcare,Provider_A,USD,168.72,NaN,5.840340e+10,208731817.0,2026-09-08 10:35:35
136,74,2026-08-01,Netflix Inc.,NFLX,Communication,Provider_B,USD,257.55,NaN,1.398965e+11,173635109.0,2026-09-08 07:34:06
199,133,2026-08-01,Wells Fargo,WFC,Financials,Provider_A,USD,636.94,6.272234e+11,1.163093e+10,NaN,2026-09-08 11:17:17
210,384,2026-08-02,AT&T,T,Communication,Provider_D,USD,282.30,9.189631e+11,3.832710e+11,NaN,2026-09-08 21:56:01
232,330,2026-08-02,Bank of America,BAC,Financials,Provider_B,USD,61.22,3.592352e+11,NaN,34954247.0,2026-09-08 15:35:02
246,268,2026-08-02,Broadcom Inc.,AVGO,Technology,Provider_D,USD,658.69,NaN,2.980465e+11,212141158.0,2026-09-08 18:35:32


In [8]:
# Check 2: Duplicate records

duplicates = df.duplicated(
    subset=["date", "company", "ticker", "provider"],
    keep=False
)

duplicate_records = df[duplicates]

print(f"Duplicate records: {len(duplicate_records):,}")
print(f"Percentage affected: {len(duplicate_records) / len(df) * 100:.2f}%")

Duplicate records: 228
Percentage affected: 2.96%


In [9]:
duplicate_records.sort_values(
    ["company", "date", "provider"]
).head(20)

,record_id,date,company,ticker,sector,provider,currency,share_price,market_cap,revenue,portfolio_value,ingestion_timestamp
0,89,2026-08-01,AMD,AMD,Technology,Provider_A,USD,102.58,1.468768e+12,4.041251e+11,79983893.0,2026-09-08 22:03:45
1,7679,2026-08-01,AMD,AMD,Technology,Provider_A,USD,102.58,1.468768e+12,4.041251e+11,79983893.0,2026-09-08 22:03:45
4,92,2026-08-01,AMD,AMD,Technology,Provider_D,USD,102.38,1.465862e+12,4.033254e+11,79825624.0,2026-09-08 09:57:50
5,7686,2026-08-01,AMD,AMD,Technology,Provider_D,USD,102.38,1.465862e+12,4.033254e+11,79825624.0,2026-09-08 09:57:50
2639,2689,2026-08-14,AMD,AMD,Technology,Provider_A,USD,108.93,1.468019e+12,4.030410e+11,348755010.0,2026-09-08 16:52:32
2640,7697,2026-08-14,AMD,AMD,Technology,Provider_A,USD,108.93,1.468019e+12,4.030410e+11,348755010.0,2026-09-08 16:52:32
6701,6690,2026-09-03,AMD,AMD,Technology,Provider_B,USD,107.65,1.469332e+12,4.036448e+11,146713248.0,2026-09-08 07:14:19
6702,7626,2026-09-03,AMD,AMD,Technology,Provider_B,USD,107.65,1.469332e+12,4.036448e+11,146713248.0,2026-09-08 07:14:19
1423,1581,2026-08-08,AT&T,T,Communication,Provider_A,USD,288.41,9.212648e+11,3.843551e+11,84764961.0,2026-09-08 14:41:34
1424,7618,2026-08-08,AT&T,T,Communication,Provider_A,USD,288.41,9.212648e+11,3.843551e+11,84764961.0,2026-09-08 14:41:34


In [11]:
# Check 3: Invalid financial values

invalid_values = (
    (df["share_price"] <= 0) |
    (df["market_cap"] <= 0) |
    (df["revenue"] <= 0) |
    (df["portfolio_value"] <= 0)
)

invalid_records = df[invalid_values]

print(f"Invalid financial records: {len(invalid_records):,}")
print(f"Percentage affected: {len(invalid_records) / len(df) * 100:.2f}%")

Invalid financial records: 114
Percentage affected: 1.48%


In [12]:
invalid_records[
    ["record_id", "date", "company", "provider",
     "share_price", "market_cap", "revenue", "portfolio_value"]
].head(20)

,record_id,date,company,provider,share_price,market_cap,revenue,portfolio_value
87,86,2026-08-01,Intel Corp.,Provider_B,194.26,-2.204377e+12,2.080692e+11,111080356.0
212,318,2026-08-02,AbbVie Inc.,Provider_B,-623.18,3.908604e+12,3.799946e+11,476641677.0
244,266,2026-08-02,Broadcom Inc.,Provider_B,661.05,-1.321473e+12,2.991104e+11,212898393.0
292,230,2026-08-02,JPMorgan Chase,Provider_B,-741.11,2.169919e+12,1.655670e+11,81397589.0
297,247,2026-08-02,Johnson & Johnson,Provider_C,601.12,-1.402133e+12,2.929238e+11,183321939.0
438,538,2026-08-03,Berkshire Hathaway,Provider_B,-492.74,1.789804e+12,4.130922e+10,121336558.0
811,891,2026-08-05,AMD,Provider_C,106.90,-1.474470e+12,4.045684e+11,264073258.0
820,920,2026-08-05,AbbVie Inc.,Provider_D,624.56,3.911802e+12,-3.800717e+11,674238355.0
904,847,2026-08-05,Johnson & Johnson,Provider_C,604.71,-1.401475e+12,2.930848e+11,186791705.0
1063,1056,2026-08-06,Chevron Corp.,Provider_D,202.06,3.982629e+12,-3.950777e+11,264784038.0


In [15]:
# Check 4: Cross-provider reconciliation
# Only compare valid, non-missing market-cap values

recon_df = df.drop_duplicates(
    subset=["date", "company", "ticker", "provider"],
    keep="first"
).copy()

# Exclude invalid/missing market caps from reconciliation
valid_market_cap = (
    recon_df["market_cap"].notna() &
    (recon_df["market_cap"] > 0)
)

recon_valid = recon_df[valid_market_cap].copy()

# Provider consensus = median of valid provider values
provider_median = (
    recon_valid
    .groupby(["date", "company", "ticker"])["market_cap"]
    .transform("median")
)

recon_valid["market_cap_diff_pct"] = (
    (recon_valid["market_cap"] - provider_median).abs()
    / provider_median
    * 100
)

# Flag significant disagreement
reconciliation_issues = recon_valid[
    recon_valid["market_cap_diff_pct"] > 5
].copy()

print(f"Reconciliation issues: {len(reconciliation_issues):,}")
print(
    f"Percentage affected: "
    f"{len(reconciliation_issues) / len(recon_valid) * 100:.2f}%"
)

Reconciliation issues: 204
Percentage affected: 2.72%


In [16]:
reconciliation_issues[
    [
        "date",
        "company",
        "ticker",
        "provider",
        "market_cap",
        "market_cap_diff_pct"
    ]
].sort_values(
    "market_cap_diff_pct",
    ascending=False
).head(20)

,date,company,ticker,provider,market_cap,market_cap_diff_pct
7694,2026-09-07,Union Pacific,UNP,Provider_A,5.122488e+12,44.470094
1405,2026-08-07,Verizon,VZ,Provider_C,5.604023e+12,44.464498
233,2026-08-02,Bank of America,BAC,Provider_C,5.189748e+11,44.288857
6654,2026-09-02,Pfizer Inc.,PFE,Provider_C,5.376000e+12,44.270360
1491,2026-08-08,Duke Energy,DUK,Provider_D,4.306178e+11,43.939998
3532,2026-08-18,Home Depot,HD,Provider_B,1.928717e+12,43.827752
3123,2026-08-16,Goldman Sachs,GS,Provider_A,5.405130e+12,43.255534
3058,2026-08-16,AbbVie Inc.,ABBV,Provider_D,5.580574e+12,42.878660
1463,2026-08-08,Broadcom Inc.,AVGO,Provider_D,1.880228e+12,42.778083
7061,2026-09-04,Pfizer Inc.,PFE,Provider_D,5.326566e+12,42.656458


In [17]:
# Check 5: Statistical anomaly detection

anomaly_df = df.sort_values(
    ["ticker", "date"]
).copy()

# Calculate daily percentage change for each company/provider
anomaly_df["price_change_pct"] = (
    anomaly_df
    .groupby(["ticker", "provider"])["share_price"]
    .pct_change() * 100
)

# Flag unusually large movements
price_anomalies = anomaly_df[
    anomaly_df["price_change_pct"].abs() > 10
].copy()

print(f"Price anomalies: {len(price_anomalies):,}")
print(
    f"Percentage affected: "
    f"{len(price_anomalies) / len(anomaly_df) * 100:.2f}%"
)

Price anomalies: 370
Percentage affected: 4.80%


/tmp/ipykernel_2459/2082170348.py:11: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change() * 100


In [18]:
price_anomalies[
    [
        "date",
        "company",
        "ticker",
        "provider",
        "share_price",
        "price_change_pct"
    ]
].sort_values(
    "price_change_pct",
    key=lambda x: x.abs(),
    ascending=False
).head(20)

,date,company,ticker,provider,share_price,price_change_pct
7040,2026-09-04,Netflix Inc.,NFLX,Provider_C,723.60,203.218237
2146,2026-08-11,Merck & Co.,MRK,Provider_D,486.12,202.595705
3929,2026-08-20,Exxon Mobil,XOM,Provider_C,46.53,-202.263736
4705,2026-08-24,Bank of America,BAC,Provider_D,190.92,202.088608
2290,2026-08-12,Cisco Systems,CSCO,Provider_C,215.31,202.019919
1272,2026-08-07,Coca-Cola,KO,Provider_A,1331.76,201.972700
1391,2026-08-07,Texas Instruments,TXN,Provider_B,153.20,-201.665671
640,2026-08-04,Berkshire Hathaway,BRK.B,Provider_B,500.75,-201.625604
3390,2026-08-17,Nike Inc.,NKE,Provider_B,-409.28,-201.606216
4232,2026-08-21,Tesla Inc.,TSLA,Provider_D,426.33,-201.567600


In [19]:
# Check 5B: Statistical anomaly detection using Z-score

anomaly_df = df.sort_values(
    ["ticker", "provider", "date"]
).copy()

# Daily percentage change
anomaly_df["price_change_pct"] = (
    anomaly_df
    .groupby(["ticker", "provider"])["share_price"]
    .pct_change(fill_method=None) * 100
)

# Calculate mean and standard deviation for each company/provider
mean_change = (
    anomaly_df
    .groupby(["ticker", "provider"])["price_change_pct"]
    .transform("mean")
)

std_change = (
    anomaly_df
    .groupby(["ticker", "provider"])["price_change_pct"]
    .transform("std")
)

# Z-score
anomaly_df["z_score"] = (
    (anomaly_df["price_change_pct"] - mean_change)
    / std_change
)

# Flag statistically unusual movements
price_anomalies = anomaly_df[
    anomaly_df["z_score"].abs() > 3
].copy()

print(f"Statistical anomalies: {len(price_anomalies):,}")
print(
    f"Percentage affected: "
    f"{len(price_anomalies) / len(anomaly_df) * 100:.2f}%"
)

Statistical anomalies: 185
Percentage affected: 2.40%


In [20]:
price_anomalies[
    [
        "date",
        "company",
        "ticker",
        "provider",
        "share_price",
        "price_change_pct",
        "z_score"
    ]
].sort_values(
    "z_score",
    key=lambda x: x.abs(),
    ascending=False
).head(20)

,date,company,ticker,provider,share_price,price_change_pct,z_score
7040,2026-09-04,Netflix Inc.,NFLX,Provider_C,723.600,203.218237,5.830606
790,2026-08-04,Union Pacific,UNP,Provider_B,124.620,183.214399,5.739004
5428,2026-08-27,PepsiCo,PEP,Provider_A,233.610,-66.439685,-5.709476
2740,2026-08-14,Lockheed Martin,LMT,Provider_B,1420.680,200.972396,5.697976
6053,2026-08-30,Qualcomm,QCOM,Provider_A,1497.900,198.964134,5.691059
7416,2026-09-06,Mastercard Inc.,MA,Provider_C,1988.370,196.877986,5.683017
3030,2026-08-15,Union Pacific,UNP,Provider_D,127.680,190.099404,5.669899
3029,2026-08-15,Union Pacific,UNP,Provider_C,127.430,189.347305,5.667168
7497,2026-09-06,Verizon,VZ,Provider_C,1701.775,153.545941,5.665933
2769,2026-08-14,Morgan Stanley,MS,Provider_B,183.075,150.170812,5.650423


In [21]:
# Check 6: Stale financial data

stale_df = df.sort_values(
    ["ticker", "provider", "date"]
).copy()

# Compare today's price with the previous reported price
stale_df["previous_price"] = (
    stale_df
    .groupby(["ticker", "provider"])["share_price"]
    .shift(1)
)

# Flag unchanged prices
stale_df["is_stale"] = (
    stale_df["share_price"].notna() &
    stale_df["previous_price"].notna() &
    (stale_df["share_price"] == stale_df["previous_price"])
)

stale_records = stale_df[stale_df["is_stale"]].copy()

print(f"Stale records: {len(stale_records):,}")
print(
    f"Percentage affected: "
    f"{len(stale_records) / len(stale_df) * 100:.2f}%"
)

Stale records: 238
Percentage affected: 3.09%


In [22]:
stale_records[
    [
        "date",
        "company",
        "ticker",
        "provider",
        "share_price",
        "previous_price"
    ]
].head(20)

,date,company,ticker,provider,share_price,previous_price
5692,2026-08-29,AbbVie Inc.,ABBV,Provider_B,622.540,622.540
1839,2026-08-10,AbbVie Inc.,ABBV,Provider_D,650.950,650.950
3461,2026-08-18,Adobe Inc.,ADBE,Provider_A,36.780,36.780
5903,2026-08-30,Adobe Inc.,ADBE,Provider_A,37.630,37.630
6105,2026-08-31,Adobe Inc.,ADBE,Provider_A,37.630,37.630
7323,2026-09-06,Adobe Inc.,ADBE,Provider_B,37.200,37.200
2044,2026-08-11,Adobe Inc.,ADBE,Provider_C,35.170,35.170
4480,2026-08-23,Adobe Inc.,ADBE,Provider_C,37.250,37.250
6107,2026-08-31,Adobe Inc.,ADBE,Provider_C,37.510,37.510
2655,2026-08-14,Adobe Inc.,ADBE,Provider_D,36.890,36.890


In [23]:
# Check 7A: Ingestion timestamp validity

df["ingestion_datetime"] = pd.to_datetime(
    df["ingestion_timestamp"],
    errors="coerce"
)

invalid_timestamps = df[
    df["ingestion_datetime"].isna()
]

print(f"Invalid timestamps: {len(invalid_timestamps):,}")
print(
    f"Percentage affected: "
    f"{len(invalid_timestamps) / len(df) * 100:.2f}%"
)

Invalid timestamps: 30
Percentage affected: 0.39%


In [24]:
# Check 7B: Currency consistency

currency_issues = df[
    ~df["currency"].isin(["USD"])
]

print(f"Currency issues: {len(currency_issues):,}")
print(
    f"Percentage affected: "
    f"{len(currency_issues) / len(df) * 100:.2f}%"
)

Currency issues: 76
Percentage affected: 0.99%


In [25]:
# ============================================================
# MASTER DATA QUALITY ENGINE
# ============================================================

quality = df.copy()

# ------------------------------------------------------------
# 1. COMPLETENESS
# ------------------------------------------------------------

quality["missing_flag"] = quality[
    ["share_price", "market_cap", "revenue", "portfolio_value"]
].isna().any(axis=1)


# ------------------------------------------------------------
# 2. UNIQUENESS
# ------------------------------------------------------------

quality["duplicate_flag"] = quality.duplicated(
    subset=["date", "company", "ticker", "provider"],
    keep=False
)


# ------------------------------------------------------------
# 3. VALIDITY
# ------------------------------------------------------------

quality["invalid_value_flag"] = (
    (quality["share_price"] <= 0) |
    (quality["market_cap"] <= 0) |
    (quality["revenue"] <= 0) |
    (quality["portfolio_value"] <= 0)
)


# ------------------------------------------------------------
# 4. RECONCILIATION
# ------------------------------------------------------------

valid_mc = (
    quality["market_cap"].notna() &
    (quality["market_cap"] > 0)
)

# Calculate provider median
quality["provider_median_market_cap"] = (
    quality[valid_mc]
    .groupby(["date", "company", "ticker"])["market_cap"]
    .transform("median")
)

quality["reconciliation_diff_pct"] = (
    (quality["market_cap"] - quality["provider_median_market_cap"]).abs()
    / quality["provider_median_market_cap"]
    * 100
)

quality["reconciliation_flag"] = (
    quality["reconciliation_diff_pct"] > 5
)


# ------------------------------------------------------------
# 5. STATISTICAL ANOMALY
# ------------------------------------------------------------

quality = quality.sort_values(
    ["ticker", "provider", "date"]
).copy()

quality["price_change_pct"] = (
    quality
    .groupby(["ticker", "provider"])["share_price"]
    .pct_change(fill_method=None) * 100
)

mean_change = (
    quality
    .groupby(["ticker", "provider"])["price_change_pct"]
    .transform("mean")
)

std_change = (
    quality
    .groupby(["ticker", "provider"])["price_change_pct"]
    .transform("std")
)

quality["z_score"] = (
    (quality["price_change_pct"] - mean_change)
    / std_change
)

quality["anomaly_flag"] = (
    quality["z_score"].abs() > 3
)


# ------------------------------------------------------------
# 6. FRESHNESS / STALE DATA
# ------------------------------------------------------------

quality["previous_price"] = (
    quality
    .groupby(["ticker", "provider"])["share_price"]
    .shift(1)
)

quality["stale_flag"] = (
    quality["share_price"].notna() &
    quality["previous_price"].notna() &
    (quality["share_price"] == quality["previous_price"])
)


# ------------------------------------------------------------
# 7. CURRENCY
# ------------------------------------------------------------

quality["currency_flag"] = (
    ~quality["currency"].isin(["USD"])
)


# ------------------------------------------------------------
# 8. TIMESTAMP
# ------------------------------------------------------------

quality["timestamp_flag"] = (
    pd.to_datetime(
        quality["ingestion_timestamp"],
        errors="coerce"
    ).isna()
)


# ------------------------------------------------------------
# 9. TOTAL ISSUE COUNT
# ------------------------------------------------------------

flag_columns = [
    "missing_flag",
    "duplicate_flag",
    "invalid_value_flag",
    "reconciliation_flag",
    "anomaly_flag",
    "stale_flag",
    "currency_flag",
    "timestamp_flag"
]

quality["issue_count"] = quality[flag_columns].sum(axis=1)


# ------------------------------------------------------------
# 10. SEVERITY
# ------------------------------------------------------------

quality["severity"] = np.select(
    [
        quality["invalid_value_flag"] |
        quality["reconciliation_flag"],

        quality["anomaly_flag"] |
        quality["missing_flag"] |
        quality["duplicate_flag"] |
        quality["stale_flag"],

        quality["currency_flag"] |
        quality["timestamp_flag"]
    ],
    [
        "Critical",
        "Warning",
        "Minor"
    ],
    default="Pass"
)


# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

print("===== DATA QUALITY SUMMARY =====")

print(f"Total records: {len(quality):,}")

print(
    f"Records with issues: "
    f"{(quality['issue_count'] > 0).sum():,}"
)

print(
    f"Clean records: "
    f"{(quality['issue_count'] == 0).sum():,}"
)

print(
    f"Overall quality rate: "
    f"{(quality['issue_count'] == 0).mean() * 100:.2f}%"
)

print("\n===== SEVERITY =====")

print(
    quality["severity"]
    .value_counts()
)

===== DATA QUALITY SUMMARY =====
Total records: 7,714
Records with issues: 1,146
Clean records: 6,568
Overall quality rate: 85.14%

===== SEVERITY =====
severity
Pass        6568
Warning      728
Critical     318
Minor        100
Name: count, dtype: int64


In [26]:
# Provider performance

provider_quality = (
    quality
    .groupby("provider")
    .agg(
        Total_Records=("record_id", "count"),
        Records_With_Issues=("issue_count", lambda x: (x > 0).sum()),
        Critical=("severity", lambda x: (x == "Critical").sum()),
        Warning=("severity", lambda x: (x == "Warning").sum()),
        Minor=("severity", lambda x: (x == "Minor").sum())
    )
    .reset_index()
)

provider_quality["Quality_Rate"] = (
    1 -
    provider_quality["Records_With_Issues"] /
    provider_quality["Total_Records"]
) * 100

provider_quality["Quality_Rate"] = provider_quality["Quality_Rate"].round(2)

provider_quality

,provider,Total_Records,Records_With_Issues,Critical,Warning,Minor,Quality_Rate
0,Provider_A,1931,273,76,176,21,85.86
1,Provider_B,1934,287,81,185,21,85.16
2,Provider_C,1926,290,77,186,27,84.94
3,Provider_D,1923,296,84,181,31,84.61


In [27]:
# Companies with the most quality issues

company_quality = (
    quality
    .groupby(["company", "ticker"])
    .agg(
        Total_Records=("record_id", "count"),
        Records_With_Issues=("issue_count", lambda x: (x > 0).sum()),
        Total_Issues=("issue_count", "sum"),
        Critical=("severity", lambda x: (x == "Critical").sum())
    )
    .reset_index()
    .sort_values("Records_With_Issues", ascending=False)
)

company_quality.head(10)

,company,ticker,Total_Records,Records_With_Issues,Total_Issues,Critical
3,Adobe Inc.,ADBE,156,34,41,8
0,AMD,AMD,156,33,37,5
8,Berkshire Hathaway,BRK.B,156,30,37,9
29,Meta Platforms,META,157,30,36,8
31,Morgan Stanley,MS,158,29,36,4
17,Duke Energy,DUK,155,29,32,8
44,UPS,UPS,155,29,33,5
30,Microsoft Corp.,MSFT,156,28,32,6
22,JPMorgan Chase,JPM,156,27,32,4
28,Merck & Co.,MRK,155,27,31,5


In [28]:
# Daily quality trend

daily_quality = (
    quality
    .groupby("date")
    .agg(
        Total_Records=("record_id", "count"),
        Records_With_Issues=("issue_count", lambda x: (x > 0).sum()),
        Critical=("severity", lambda x: (x == "Critical").sum()),
        Warning=("severity", lambda x: (x == "Warning").sum())
    )
    .reset_index()
)

daily_quality["Quality_Rate"] = (
    1 -
    daily_quality["Records_With_Issues"] /
    daily_quality["Total_Records"]
) * 100

daily_quality["Quality_Rate"] = daily_quality["Quality_Rate"].round(2)

daily_quality

,date,Total_Records,Records_With_Issues,Critical,Warning,Quality_Rate
0,2026-08-01,203,24,5,13,88.18
1,2026-08-02,202,36,12,20,82.18
2,2026-08-03,202,23,3,18,88.61
3,2026-08-04,202,29,9,16,85.64
4,2026-08-05,202,22,6,11,89.11
5,2026-08-06,205,35,8,23,82.93
6,2026-08-07,203,30,8,19,85.22
7,2026-08-08,206,34,10,24,83.50
8,2026-08-09,202,26,6,17,87.13
9,2026-08-10,203,24,7,14,88.18


In [30]:
# ============================================================
# FINAL POWER BI DATASETS
# ============================================================

# ------------------------------------------------------------
# 1. RECORD-LEVEL QUALITY DATA
# ------------------------------------------------------------

final_quality = quality.copy()

def get_issue_types(row):
    issues = []

    if row["missing_flag"]:
        issues.append("Missing Value")

    if row["duplicate_flag"]:
        issues.append("Duplicate")

    if row["invalid_value_flag"]:
        issues.append("Invalid Financial Value")

    if row["reconciliation_flag"]:
        issues.append("Provider Reconciliation")

    if row["anomaly_flag"]:
        issues.append("Statistical Anomaly")

    if row["stale_flag"]:
        issues.append("Stale Data")

    if row["currency_flag"]:
        issues.append("Currency Inconsistency")

    if row["timestamp_flag"]:
        issues.append("Invalid Timestamp")

    return ", ".join(issues) if issues else "No Issue"


final_quality["issue_types"] = final_quality.apply(
    get_issue_types,
    axis=1
)

final_quality = final_quality[
    [
        "record_id",
        "date",
        "company",
        "ticker",
        "sector",
        "provider",
        "currency",
        "share_price",
        "market_cap",
        "revenue",
        "portfolio_value",
        "issue_types",
        "issue_count",
        "severity"
    ]
].copy()


# ------------------------------------------------------------
# 2. ISSUE SUMMARY
# ------------------------------------------------------------

issue_summary = pd.DataFrame({
    "Issue_Type": [
        "Missing Values",
        "Duplicate Records",
        "Invalid Financial Values",
        "Provider Reconciliation",
        "Statistical Anomaly",
        "Stale Data",
        "Currency Inconsistency",
        "Invalid Timestamp"
    ],
    "Affected_Records": [
        quality["missing_flag"].sum(),
        quality["duplicate_flag"].sum(),
        quality["invalid_value_flag"].sum(),
        quality["reconciliation_flag"].sum(),
        quality["anomaly_flag"].sum(),
        quality["stale_flag"].sum(),
        quality["currency_flag"].sum(),
        quality["timestamp_flag"].sum()
    ]
})

issue_summary["Percentage_of_Records"] = (
    issue_summary["Affected_Records"] /
    len(quality) * 100
).round(2)


# ------------------------------------------------------------
# 3. PROVIDER QUALITY
# ------------------------------------------------------------

provider_quality = (
    quality
    .groupby("provider")
    .agg(
        Total_Records=("record_id", "count"),
        Records_With_Issues=(
            "issue_count",
            lambda x: (x > 0).sum()
        ),
        Critical=(
            "severity",
            lambda x: (x == "Critical").sum()
        ),
        Warning=(
            "severity",
            lambda x: (x == "Warning").sum()
        ),
        Minor=(
            "severity",
            lambda x: (x == "Minor").sum()
        )
    )
    .reset_index()
)

provider_quality["Quality_Rate"] = (
    1 -
    provider_quality["Records_With_Issues"] /
    provider_quality["Total_Records"]
) * 100

provider_quality["Quality_Rate"] = (
    provider_quality["Quality_Rate"].round(2)
)


# ------------------------------------------------------------
# 4. DAILY QUALITY
# ------------------------------------------------------------

daily_quality = (
    quality
    .groupby("date")
    .agg(
        Total_Records=("record_id", "count"),
        Records_With_Issues=(
            "issue_count",
            lambda x: (x > 0).sum()
        ),
        Critical=(
            "severity",
            lambda x: (x == "Critical").sum()
        ),
        Warning=(
            "severity",
            lambda x: (x == "Warning").sum()
        )
    )
    .reset_index()
)

daily_quality["Quality_Rate"] = (
    1 -
    daily_quality["Records_With_Issues"] /
    daily_quality["Total_Records"]
) * 100

daily_quality["Quality_Rate"] = (
    daily_quality["Quality_Rate"].round(2)
)


# ------------------------------------------------------------
# 5. COMPANY QUALITY
# ------------------------------------------------------------

company_quality = (
    quality
    .groupby(["company", "ticker"])
    .agg(
        Total_Records=("record_id", "count"),
        Records_With_Issues=(
            "issue_count",
            lambda x: (x > 0).sum()
        ),
        Total_Issues=("issue_count", "sum"),
        Critical=(
            "severity",
            lambda x: (x == "Critical").sum()
        )
    )
    .reset_index()
    .sort_values(
        "Records_With_Issues",
        ascending=False
    )
)


# ------------------------------------------------------------
# 6. EXPORT EVERYTHING
# ------------------------------------------------------------

final_quality.to_csv(
    "arcana_final_quality_data.csv",
    index=False
)

issue_summary.to_csv(
    "arcana_issue_summary.csv",
    index=False
)

provider_quality.to_csv(
    "arcana_provider_quality.csv",
    index=False
)

daily_quality.to_csv(
    "arcana_daily_quality.csv",
    index=False
)

company_quality.to_csv(
    "arcana_company_quality.csv",
    index=False
)


# ------------------------------------------------------------
# 7. DISPLAY RESULTS
# ------------------------------------------------------------

print("✅ ALL POWER BI DATASETS CREATED")
print(f"Records: {len(final_quality):,}")

print("\n===== ISSUE SUMMARY =====")
display(issue_summary)

print("\n===== PROVIDER QUALITY =====")
display(provider_quality)

print("\n===== TOP COMPANIES BY ISSUES =====")
display(company_quality.head(10))

✅ ALL POWER BI DATASETS CREATED
Records: 7,714

===== ISSUE SUMMARY =====


,Issue_Type,Affected_Records,Percentage_of_Records
0,Missing Values,228,2.96
1,Duplicate Records,228,2.96
2,Invalid Financial Values,114,1.48
3,Provider Reconciliation,204,2.64
4,Statistical Anomaly,185,2.40
5,Stale Data,238,3.09
6,Currency Inconsistency,76,0.99
7,Invalid Timestamp,30,0.39



===== PROVIDER QUALITY =====


,provider,Total_Records,Records_With_Issues,Critical,Warning,Minor,Quality_Rate
0,Provider_A,1931,273,76,176,21,85.86
1,Provider_B,1934,287,81,185,21,85.16
2,Provider_C,1926,290,77,186,27,84.94
3,Provider_D,1923,296,84,181,31,84.61



===== TOP COMPANIES BY ISSUES =====


,company,ticker,Total_Records,Records_With_Issues,Total_Issues,Critical
3,Adobe Inc.,ADBE,156,34,41,8
0,AMD,AMD,156,33,37,5
8,Berkshire Hathaway,BRK.B,156,30,37,9
29,Meta Platforms,META,157,30,36,8
31,Morgan Stanley,MS,158,29,36,4
17,Duke Energy,DUK,155,29,32,8
44,UPS,UPS,155,29,33,5
30,Microsoft Corp.,MSFT,156,28,32,6
22,JPMorgan Chase,JPM,156,27,32,4
28,Merck & Co.,MRK,155,27,31,5
